In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import classification_report, roc_auc_score 

In [5]:
# load the scaled dataset
df_logreg = pd.read_csv("../data/processed/features_2023_scaled.csv")

In [15]:
print(f"Shape of the dataset: {df_logreg.shape}\n\n")
print("Races available: ",  df_logreg["RaceName"].unique())

Shape of the dataset: (11089, 27)


Races available:  ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Monza'
 'Saudi Arabia' 'Silverstone' 'Singapore' 'Spain']


```
Why we split by race, not randomly 

train, test = train_test_split(df_logreg, test_size = 0.2, random_state 42)

a random split picks rows completely randomly, regardless of which race they came from. This means we'd likely end up with laps from the SAME race in BOTH training set and test set.


DATA LEAKAGe!
laps from the same race are not independent of each other.They share context, same track temperature, same safety care periods, same overall race strategy patterns, similar tyre degradation curves for that specific circuit.

If the model sees SOME laps from Bahrain during training, then gets tested on OTHER laps from the SAME bahrain race, it's not really being tested on 'unseen' data. It is already partially learned that specific race's conditions and patterns. This makes the test result look artificially better than they actually are

This is called a data leakage, information leaking between train and test sets that shouldn't be there in a fair evaluation.
```


 ----

```

Choosing test races:

Abu Dhabi     → modern, medium-speed circuit, lots of overtaking zones
Australia     → semi-street circuit, moderate degradation
Bahrain       → high degradation, desert conditions
Hungary       → very high degradation, low overtaking
Monaco        → street circuit, lowest degradation, no real pit strategy battles
Monza         → very low degradation, high speed
Saudi Arabia  → street circuit, high speed
Silverstone   → classic circuit, high speed corners
Singapore     → street circuit, night race, high safety car probability
Spain         → medium degradation, technical circuit



Singapore → street circuit, high safety car chance, different strategy dynamics
Monza     → very different track type, low degradation, high speed

Together they represent genuinely DIFFERENT racing conditions
from most of our training races, which makes for a fairer,
more challenging test of whether our model truly generalizes
```

 -----

In [36]:
# Split races 
test_races = ["Singapore", "Monza"]

# give me everything NOT in this list
train_df = df_logreg[~df_logreg["RaceName"].isin(test_races)]
test_df = df_logreg[df_logreg["RaceName"].isin(test_races)]

print(f"\n\n Train shape: {train_df.shape}")
print(f"Test shape:       {test_df.shape}\n\n")

print(f"Train races: {train_df['RaceName'].unique()}")
print(f"Test races:  {test_df['RaceName'].unique()}")


                                        



 Train shape: (9079, 27)
Test shape:       (2010, 27)


Train races: ['Abu Dhabi' 'Australia' 'Bahrain' 'Hungary' 'Monaco' 'Saudi Arabia'
 'Silverstone' 'Spain']
Test races:  ['Monza' 'Singapore']


In [46]:
# Seperate Features (X) from Target (y)

# Pitted = target, we are predicting this
# Driver, RaceName, Compound = identity/text columns, not used as model inputs

exclude_cols = ["Pitted", "Driver", "RaceName", "Compound",
               "IsAccurate", "FastF1Generated", "IsPersonalBest"]

feature_cols = [col for col in train_df.columns if col  not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df["Pitted"]

X_test = test_df[feature_cols]
y_test = test_df["Pitted"]


print(f"Feature columns: {feature_cols}")
print(f"\n X_train shape: {X_train.shape}")
print(f"\n y_train shape: {y_train.shape}")

Feature columns: ['Stint', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreLife', 'FreshTyre', 'TrackStatus', 'Position', 'CompoundEncoded', 'LapTimeDelta', 'LapTimeRolling3', 'DegradationFromStintStart', 'TotalLaps', 'RacePctComplete', 'LapsRemaining', 'IsLateRace']

 X_train shape: (9079, 20)

 y_train shape: (9079,)


In [48]:
# Training the model

# class_weight = 'balanced', handles our severe class imbalance
# 33 to 1 ratio, without this model would just predict 'never pit'


model = LogisticRegression(class_weight = 'balanced', max_iter = 1000, random_state = 42)


model.fit(X_train, y_train)
print("Model trained")


Model trained


In [52]:
# Predictions

# get predictions on the unseen test races
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("Logistic Regression Performance (Singapore + Monza, unseen):\n")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {auc:.4f}")


Logistic Regression Performance (Singapore + Monza, unseen):

              precision    recall  f1-score   support

           0       1.00      0.38      0.55      1959
           1       0.04      1.00      0.08        51

    accuracy                           0.39      2010
   macro avg       0.52      0.69      0.31      2010
weighted avg       0.98      0.39      0.54      2010

ROC-AUC: 0.8362
